# H&M Future Repurchase Propensity — Logistic Regression

This analysis predicts whether Recent Occasional customers will purchase again during the future 120-day outcome window.

An RFM baseline model is compared with expanded behavioural specifications incorporating customer engagement, membership status, product exploration, basket depth, and channel behaviour.

Model specifications are compared on a validation set. The selected specification is then refitted on the combined training and validation data and evaluated once on an untouched test set.

The objective is to determine whether these behavioural signals improve CRM targeting beyond historical RFM alone.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("/Volumes/USB/H&M Data/recent_occasional_logistic_regression_input.csv")

In [3]:
df.shape

(328787, 12)

## Initial Logistic-Regression Specification

### Dependent Variable

- `repurchase`

The target indicates whether a Recent Occasional customer purchased again during the future 120-day outcome window.

### Historical RFM Controls

- `recency`
- `log_frequency`
- `log_monetary`

### Core Behavioural Signals

- `regular_fashion_news`
- `club_member_active`
- `product_group_entropy`
- `both_channels`

### Additional Specification Candidates

- `items_per_shopping_day`
- `channel_2_item_share`

The analysis first tests whether the core behavioural signals improve prediction beyond historical RFM. It then evaluates whether basket depth and Channel 2 purchasing concentration provide additional predictive value.

In [4]:
initial_features = ["recency","log_frequency","log_monetary",
                    "regular_fashion_news","club_member_active","product_group_entropy","both_channels",
                   "items_per_shopping_day","channel_2_item_share"]

In [5]:
X = df[initial_features]
y = df["repurchase"]

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_development, X_test, y_development, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

X_train, X_validation, y_train, y_validation = train_test_split(X_development,y_development,test_size=0.25,random_state=42,stratify=y_development)

In [8]:
print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)
print(y_train.mean())
print(y_validation.mean())
print(y_test.mean())

(197271, 9)
(65758, 9)
(65758, 9)
0.3791079276730994
0.3791021624745278
0.3791021624745278


## Variable Preprocessing

Continuous variables are standardized using the mean and standard deviation of the training dataset during model selection.

Binary variables remain coded as 0 and 1.

The validation and test datasets are not used to fit this initial scaler. After model selection, a new scaler is fitted on the combined training and validation data for final model estimation.

In [9]:
continuous_features = ["recency","log_frequency","log_monetary","product_group_entropy","items_per_shopping_day","channel_2_item_share"]

In [10]:
binary_features = ["regular_fashion_news","club_member_active","both_channels"]

In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
X_train_scaled = X_train.copy()
X_validation_scaled = X_validation.copy()
X_test_scaled = X_test.copy()

In [13]:
scaler = StandardScaler()
scaler.fit(X_train[continuous_features])

,copy,True
,with_mean,True
,with_std,True


In [14]:
X_train_scaled[continuous_features] = scaler.transform(X_train[continuous_features])

In [15]:
X_validation_scaled[continuous_features] = scaler.transform(X_validation[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

In [16]:
display(X_train_scaled[binary_features].drop_duplicates())

,regular_fashion_news,club_member_active,both_channels
25339,0,1,1
235552,0,1,0
154701,1,1,0
178429,0,0,0
71567,1,1,1
108535,1,0,0
232078,0,0,1
25776,1,0,1


In [17]:
display(X_train_scaled[continuous_features].agg(["mean", "std"]).round(3))

,recency,log_frequency,log_monetary,product_group_entropy,items_per_shopping_day,channel_2_item_share
mean,-0.0,0.0,0.0,0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0


# Logistic-Regression Models

The modelling process begins with two core models, followed by additional feature testing.

## RFM Baseline Model

Uses only customers’ historical Recency, Frequency, and Monetary behaviour.

## Expanded Behavioural Model

Adds CRM engagement, membership status, product exploration, and cross-channel adoption to the RFM baseline.

## Additional Feature Testing

Two additional behavioural variables are evaluated:

- **Items per shopping day** represents basket depth rather than shopping frequency.
- **Channel 2 item share** represents the proportion of purchased items coming from Channel 2 and captures channel preference.

These variables are tested individually and jointly to determine whether they provide meaningful predictive value beyond the original expanded model.

Comparing the model specifications shows whether behavioural and channel signals improve the prediction of future 120-day repurchase beyond customers’ historical RFM behaviour.

In [18]:
from sklearn.linear_model import LogisticRegression

In [19]:
baseline_features = ["recency","log_frequency","log_monetary"]

In [20]:
baseline_model = LogisticRegression(max_iter=1000)

In [21]:
baseline_model.fit(X_train_scaled[baseline_features],y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
expanded_features = ["recency","log_frequency","log_monetary",
    "regular_fashion_news","club_member_active","product_group_entropy","both_channels"]

In [23]:
expanded_model = LogisticRegression(max_iter=1000)

In [24]:
expanded_model.fit(X_train_scaled[expanded_features],y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [25]:
basket_depth_features = expanded_features + ["items_per_shopping_day"]

In [26]:
basket_depth_model = LogisticRegression(max_iter=1000)
basket_depth_model.fit(X_train_scaled[basket_depth_features],y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [27]:
channel_preference_features = expanded_features + ["channel_2_item_share"]

In [28]:
channel_preference_model = LogisticRegression(max_iter=1000)
channel_preference_model.fit(X_train_scaled[channel_preference_features],y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [29]:
full_candidate_features = expanded_features + ["items_per_shopping_day","channel_2_item_share"]

In [30]:
full_candidate_model = LogisticRegression(max_iter=1000)
full_candidate_model.fit(X_train_scaled[full_candidate_features],y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Model Performance Comparison

The RFM baseline, expanded behavioural, and three additional feature specifications are evaluated on the same validation set using ROC AUC and PR AUC.

The untouched test set is not used during this model-selection stage.

In [31]:
baseline_validation_probability = baseline_model.predict_proba(X_validation_scaled[baseline_features])[:, 1]

In [32]:
expanded_validation_probability = expanded_model.predict_proba(X_validation_scaled[expanded_features])[:, 1]

In [33]:
basket_depth_validation_probability = basket_depth_model.predict_proba(X_validation_scaled[basket_depth_features])[:,1]

In [34]:
channel_preference_validation_probability = channel_preference_model.predict_proba(X_validation_scaled[channel_preference_features])[:,1]

In [35]:
full_candidate_validation_probability = full_candidate_model.predict_proba(X_validation_scaled[full_candidate_features])[:,1]

In [36]:
from sklearn.metrics import roc_auc_score, average_precision_score

In [37]:
baseline_roc_auc = roc_auc_score(y_validation, baseline_validation_probability)
expanded_roc_auc = roc_auc_score(y_validation, expanded_validation_probability)
basket_depth_roc_auc = roc_auc_score(y_validation,basket_depth_validation_probability)
channel_preference_roc_auc = roc_auc_score(y_validation,channel_preference_validation_probability)
full_candidate_roc_auc = roc_auc_score(y_validation,full_candidate_validation_probability)

In [38]:
baseline_pr_auc = average_precision_score(y_validation, baseline_validation_probability)
expanded_pr_auc = average_precision_score(y_validation, expanded_validation_probability)
basket_depth_pr_auc = average_precision_score(y_validation,basket_depth_validation_probability)
channel_preference_pr_auc = average_precision_score(y_validation,channel_preference_validation_probability)
full_candidate_pr_auc = average_precision_score(y_validation,full_candidate_validation_probability)

In [39]:
model_comparison = pd.DataFrame({
    "Model": ["RFM Baseline", "Expanded Behavioral","Expanded + Basket Depth","Expanded + Channel Preference","Expanded + Both Candidates"],
    "ROC AUC": [baseline_roc_auc, expanded_roc_auc,basket_depth_roc_auc,channel_preference_roc_auc,full_candidate_roc_auc],
    "PR AUC": [baseline_pr_auc, expanded_pr_auc,basket_depth_pr_auc,channel_preference_pr_auc,full_candidate_pr_auc]
})

In [40]:
display(model_comparison.round(4))

,Model,ROC AUC,PR AUC
0,RFM Baseline,0.6673,0.5378
1,Expanded Behavioral,0.6846,0.5547
2,Expanded + Basket Depth,0.6848,0.5551
3,Expanded + Channel Preference,0.6895,0.5631
4,Expanded + Both Candidates,0.6896,0.5632


### Validation Performance Result

On the validation set, the **Expanded + Channel Preference** model achieved a ROC AUC of **0.6895** and PR AUC of **0.5631**, compared with **0.6673** and **0.5378** for the RFM baseline and **0.6846** and **0.5547** for the original expanded model.

Adding `channel_2_item_share` improved both ranking measures, showing that the strength of customers’ channel preference provides additional predictive information beyond RFM, CRM engagement, membership status, product exploration, and cross-channel adoption.

The specification containing both additional candidates was only marginally higher at **0.6896 ROC AUC** and **0.5632 PR AUC**. Because `items_per_shopping_day` provided negligible incremental improvement, the simpler **Expanded + Channel Preference** specification was selected.

## Final Model Refit and Untouched-Test Evaluation

The validation results select the **Expanded + Channel Preference** specification.

The selected model and the RFM baseline are now refitted on the combined training and validation data. The untouched test set is used once to estimate final model performance.

In [41]:
X_development = pd.concat([X_train,X_validation])
y_development = pd.concat([y_train,y_validation])

In [42]:
X_development_scaled = X_development.copy()
X_test_final_scaled = X_test.copy()

In [43]:
final_scaler = StandardScaler()
final_scaler.fit(X_development[continuous_features])

,copy,True
,with_mean,True
,with_std,True


In [44]:
X_development_scaled[continuous_features] = final_scaler.transform(X_development[continuous_features])

In [45]:
X_test_final_scaled[continuous_features] = final_scaler.transform(X_test[continuous_features])

In [46]:
final_baseline_model = LogisticRegression(max_iter=1000)

In [47]:
final_baseline_model.fit(X_development_scaled[baseline_features],y_development)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [48]:
final_selected_model = LogisticRegression(max_iter=1000)

In [49]:
final_selected_model.fit(X_development_scaled[channel_preference_features],y_development)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [50]:
baseline_probability = final_baseline_model.predict_proba(X_test_final_scaled[baseline_features])[:,1]

In [51]:
channel_preference_probability = final_selected_model.predict_proba(X_test_final_scaled[channel_preference_features])[:,1]

In [52]:
final_test_comparison = pd.DataFrame({
    "Model": ["RFM Baseline", "Expanded + Channel Preference"],
    "ROC AUC": [roc_auc_score(y_test,baseline_probability),roc_auc_score(y_test,channel_preference_probability)],
    "PR AUC": [average_precision_score(y_test,baseline_probability),average_precision_score(y_test,channel_preference_probability)]
})

In [53]:
display(final_test_comparison.round(4))

,Model,ROC AUC,PR AUC
0,RFM Baseline,0.6640,0.5319
1,Expanded + Channel Preference,0.6851,0.5559


### Final Test Performance Result

On the untouched test set, the selected **Expanded + Channel Preference** model achieved a ROC AUC of **0.6851** and PR AUC of **0.5559**, compared with **0.6640** and **0.5319** for the refitted RFM baseline.

These results confirm on previously unused customers that behavioural and channel signals improve future-repurchase prediction beyond historical RFM alone.

## Decile Lift Analysis

Customers are ranked by predicted repurchase probability and divided into ten equally sized groups. Decile 1 contains the highest-propensity customers, while Decile 10 contains the lowest-propensity customers.

In [54]:
overall_repurchase_rate = y_test.mean()

In [55]:
baseline_lift = pd.DataFrame({"actual": y_test.to_numpy(),"probability": baseline_probability})

In [56]:
baseline_rank = baseline_lift["probability"].rank(method="first",ascending=False)

In [57]:
baseline_lift["decile"] = pd.qcut(baseline_rank,10,labels=range(1, 11))

In [58]:
baseline_deciles = baseline_lift.groupby("decile",observed=True).agg(
    customers=("actual", "size"),
    repurchase_rate=("actual", "mean"),
    average_probability=("probability", "mean")
).reset_index()

In [59]:
baseline_deciles["lift"] = (baseline_deciles["repurchase_rate"]/ overall_repurchase_rate)

In [60]:
channel_preference_lift = pd.DataFrame({"actual": y_test.to_numpy(),"probability": channel_preference_probability})

In [61]:
channel_preference_rank = channel_preference_lift["probability"].rank(method="first",ascending=False)

In [62]:
channel_preference_lift["decile"] = pd.qcut(channel_preference_rank,10,labels=range(1, 11))

In [63]:
channel_preference_deciles = channel_preference_lift.groupby("decile",observed=True).agg(
    customers=("actual", "size"),
    repurchase_rate=("actual", "mean"),
    average_probability=("probability", "mean")
).reset_index()

In [64]:
channel_preference_deciles["lift"] = (channel_preference_deciles["repurchase_rate"]/ overall_repurchase_rate)

In [65]:
lift_comparison = baseline_deciles[["decile", "repurchase_rate", "lift"]].merge(channel_preference_deciles[["decile", "repurchase_rate", "lift"]],on="decile",suffixes=("_rfm", "_channel_preference"))

In [66]:
display(lift_comparison.round(3))

,decile,repurchase_rate_rfm,lift_rfm,repurchase_rate_channel_preference,lift_channel_preference
0,1,0.610,1.608,0.653,1.721
1,2,0.539,1.422,0.560,1.476
2,3,0.513,1.354,0.501,1.321
3,4,0.429,1.131,0.441,1.164
4,5,0.397,1.046,0.394,1.039
5,6,0.361,0.953,0.330,0.870
6,7,0.261,0.689,0.283,0.745
7,8,0.234,0.618,0.245,0.647
8,9,0.239,0.631,0.221,0.584
9,10,0.207,0.547,0.164,0.432


### Lift Result

On the untouched test set, the selected **Expanded + Channel Preference** model identified a top decile with a **65.3% repurchase rate**, representing **1.721× lift** over the customer average.

The bottom decile had a repurchase rate of only **16.4%**. The selected model created a **48.9-percentage-point difference** between the highest- and lowest-propensity groups, compared with **40.2 percentage points** for the refitted RFM baseline.

This stronger separation makes the selected model more useful for differentiated CRM targeting.

## Probability Accuracy

Brier score measures the accuracy of predicted probabilities. Lower values indicate better probability estimates.

In [67]:
from sklearn.metrics import brier_score_loss

In [68]:
baseline_brier = brier_score_loss(y_test,baseline_probability)

In [69]:
channel_preference_brier = brier_score_loss(y_test,channel_preference_probability)

In [70]:
brier_comparison = pd.DataFrame({
    "Model": ["RFM Baseline", "Expanded + Channel Preference"],
    "Brier Score": [baseline_brier, channel_preference_brier]
})

In [71]:
display(brier_comparison.round(4))

,Model,Brier Score
0,RFM Baseline,0.2173
1,Expanded + Channel Preference,0.2122


### Probability-Accuracy Result

On the untouched test set, the selected **Expanded + Channel Preference** model reduced the Brier score from **0.2173** to **0.2122**, indicating slightly more accurate probability estimates than the refitted RFM baseline.

## Selected Model Interpretation

The **Expanded + Channel Preference** specification was selected using validation performance and then refitted on the combined training and validation data.

The following coefficients describe how each feature is associated with future 120-day repurchase propensity after accounting for the other variables in the final model.

Continuous variables are standardized, while binary variables remain coded as 0 and 1.

In [72]:
coefficient_table = pd.DataFrame({"feature": channel_preference_features,"coefficient": final_selected_model.coef_[0]})

In [73]:
display(coefficient_table.round(3))

,feature,coefficient
0,recency,-0.233
1,log_frequency,0.406
2,log_monetary,0.025
3,regular_fashion_news,0.326
4,club_member_active,0.736
5,product_group_entropy,0.121
6,both_channels,0.234
7,channel_2_item_share,-0.181


In [74]:
channel_preference_intercept = final_selected_model.intercept_[0]

In [75]:
print("Intercept:", round(channel_preference_intercept, 3))

Intercept: -1.378


### Estimated Logistic-Regression Equation

$$
\begin{aligned}
\operatorname{logit}(p) ={}&
-1.378
-0.233\,z(\text{recency})
+0.406\,z(\text{log frequency}) \\
&+0.025\,z(\text{log monetary})
+0.326(\text{regular fashion news}) \\
&+0.736(\text{active club member})
+0.121\,z(\text{product-group entropy}) \\
&+0.234(\text{both channels})
-0.181\,z(\text{Channel 2 item share})
\end{aligned}
$$

Here, $p$ represents the probability that a Recent Occasional customer purchases again during the future 120-day outcome window. The notation $z(\cdot)$ indicates a continuous variable standardized using the combined training and validation data, while the remaining behavioural variables are binary indicators coded as 0 or 1.

### Average Probability Effects

To make the model easier to interpret:

- Continuous-variable effects represent the average change in predicted repurchase probability associated with a one-standard-deviation increase.
- Binary-variable effects represent the average probability difference when the feature changes from 0 to 1 while the remaining model inputs are held constant.

In [76]:
selected_continuous_features = [
    "recency",
    "log_frequency",
    "log_monetary",
    "product_group_entropy",
    "channel_2_item_share"
]

In [77]:
selected_binary_features = [
    "regular_fashion_news",
    "club_member_active",
    "both_channels"
]

In [78]:
continuous_effects = []

In [79]:
for feature in selected_continuous_features:

    changed_data = X_test_final_scaled[channel_preference_features].copy()

    changed_data[feature] = changed_data[feature] + 1

    changed_probability = final_selected_model.predict_proba(changed_data)[:,1]

    probability_change = (changed_probability - channel_preference_probability).mean()

    continuous_effects.append([feature,probability_change])

In [80]:
binary_effects = []

In [81]:
for feature in selected_binary_features:

    feature_on = X_test_final_scaled[channel_preference_features].copy()
    feature_off = X_test_final_scaled[channel_preference_features].copy()

    feature_on[feature] = 1
    feature_off[feature] = 0

    probability_on = final_selected_model.predict_proba(feature_on)[:,1]

    probability_off = final_selected_model.predict_proba(feature_off)[:,1]

    probability_change = (probability_on - probability_off).mean()

    binary_effects.append([feature,probability_change])

In [82]:
continuous_effects = pd.DataFrame(continuous_effects,columns=["feature","average_probability_change"])

In [83]:
binary_effects = pd.DataFrame(binary_effects,columns=["feature", "average_probability_change"])

In [84]:
probability_effects = pd.concat([continuous_effects, binary_effects],ignore_index=True)

In [85]:
probability_effects["probability_point_change"] = (probability_effects["average_probability_change"] * 100)

In [86]:
probability_effects["absolute_effect"] = (probability_effects["probability_point_change"].abs())

In [87]:
display(probability_effects[["feature", "probability_point_change"]].round(2))

,feature,probability_point_change
0,recency,-4.81
1,log_frequency,8.87
2,log_monetary,0.53
3,product_group_entropy,2.60
4,channel_2_item_share,-3.76
5,regular_fashion_news,7.04
6,club_member_active,14.33
7,both_channels,5.09


### Key Predictive Signals

Active club membership emerged as the strongest behavioural signal, associated with an average **14.33-percentage-point increase** in predicted repurchase probability.

Other positive signals included:

- Higher historical shopping frequency: **+8.87 percentage points**
- Regular Fashion News engagement: **+7.04 percentage points**
- Purchasing through both channels: **+5.09 percentage points**
- Greater product-group diversity: **+2.60 percentage points**

A one-standard-deviation increase in recency was associated with a **4.81-percentage-point decrease** in predicted repurchase probability.

A greater concentration of purchases in Channel 2 was associated with a **3.76-percentage-point decrease** in predicted repurchase probability after accounting for the other model variables.

Historical monetary value added little incremental predictive value after the other customer behaviours were considered.

## Logistic-Regression Conclusion

The **Expanded + Channel Preference** specification was selected on the validation set and then confirmed on the untouched test set after final refitting.

It outperformed the refitted RFM baseline across ROC AUC, PR AUC, decile lift, and probability accuracy.

It therefore becomes the primary interpretable model for CRM targeting and the benchmark for the next stage, where a nonlinear challenger will test whether threshold effects and behavioural interactions can further improve future-repurchase prediction.

In [88]:
tableau_scores = df.loc[X_test.index, [
    "customer_id",
    "segment",
    "repurchase",
    "recency",
    "log_frequency",
    "log_monetary",
    "regular_fashion_news",
    "club_member_active",
    "product_group_entropy",
    "both_channels",
    "items_per_shopping_day",
    "channel_2_item_share"
]].copy()

In [89]:
tableau_scores["predicted_probability"] = channel_preference_probability

In [90]:
tableau_scores = tableau_scores.rename(columns={"repurchase": "actual_repurchase"})

In [91]:
probability_rank = tableau_scores["predicted_probability"].rank(method="first",ascending=False)
tableau_scores["propensity_decile"] = pd.qcut(probability_rank,10,labels=range(1, 11)).astype(int)

In [92]:
tableau_scores["historical_shopping_days"] = np.rint(np.expm1(tableau_scores["log_frequency"])).astype(int)
tableau_scores["historical_spend_index"] = np.expm1(tableau_scores["log_monetary"])

In [93]:
tableau_scores["propensity_tier"] = pd.cut(tableau_scores["propensity_decile"],bins=[0, 3, 7, 10],
    labels=["High Propensity","Medium Propensity","Low Propensity"])

In [94]:
tableau_scores = tableau_scores[[
    "customer_id",
    "segment",
    "actual_repurchase",
    "predicted_probability",
    "propensity_decile",
    "propensity_tier",
    "recency",
    "historical_shopping_days",
    "historical_spend_index",
    "log_frequency",
    "log_monetary",
    "regular_fashion_news",
    "club_member_active",
    "product_group_entropy",
    "both_channels",
    "channel_2_item_share",
    "items_per_shopping_day"
]]

In [95]:
print(tableau_scores.shape)
print(tableau_scores["actual_repurchase"].mean())

display(tableau_scores.head())

(65758, 17)
0.3791021624745278


,customer_id,segment,actual_repurchase,predicted_probability,propensity_decile,propensity_tier,recency,historical_shopping_days,historical_spend_index,log_frequency,log_monetary,regular_fashion_news,club_member_active,product_group_entropy,both_channels,channel_2_item_share,items_per_shopping_day
316828,f6ab946dadebf67085ad09661d7a83f6dbf2261ed9deab...,2,1,0.352592,6,Medium Propensity,157,2,0.127068,1.098612,0.119619,0,1,0.000000,1,0.333333,1.5
284187,dd2a5416a321bafad01e36ceef30f45a08d195f3fc160d...,2,0,0.296174,7,Medium Propensity,55,1,0.283153,0.693147,0.249320,0,1,1.197340,0,1.000000,14.0
39377,1eb692d8225bc4536c229b35b8ebf6dff3992b85048c5d...,2,0,0.551839,2,High Propensity,151,4,0.117797,1.609438,0.111359,1,1,1.242453,0,1.000000,1.5
289972,e1abe9e0578492822cf5ae2c1d84c35bca4c3b665e0a83...,2,0,0.257136,8,Low Propensity,128,1,0.050814,0.693147,0.049565,1,1,0.693147,0,1.000000,2.0
287159,df6aa3667bd9da671f2c6b629910f31756cbfcdf1e7b98...,2,1,0.322278,6,Medium Propensity,108,2,0.189712,1.098612,0.173711,0,1,0.598270,0,1.000000,3.5


In [96]:
len(X_test)

65758

In [97]:
assert len(tableau_scores) == len(X_test)
assert tableau_scores["customer_id"].is_unique
assert tableau_scores["predicted_probability"].between(0, 1).all()

In [98]:
tableau_scores.to_csv("/Volumes/USB/H&M Data/tableau_recent_occasional_scores.csv",index=False)

In [99]:
print("Saved tableau_recent_occasional_scores.csv")

Saved tableau_recent_occasional_scores.csv
